# Model-size LLR-prior popDMS report

This notebook is the reporting layer for the ESM-C model-size benchmark. It does
**not** run inference: `experiments/model_size_llr/submit_experiment.sh` launches
the whole pipeline on the cluster (process → GPU LLR for ESM-C 300M/600M/6B →
per-dataset `esmdms analyze` → aggregate), and this notebook loads the aggregated
`baselines.csv`, `prior_sweeps.csv`, and `summary.csv` and plots them.

Point it at a different results set with the `ESMDMS_AGGREGATE_DIR` environment
variable; it defaults to `results/model_size_llr/aggregate`.

**What is compared, per dataset:** the naive enrichment ratio, the assay's own DMS
functional score (where available), regular popDMS (no prior, gamma at the popDMS
correlation elbow), the raw ESM-C LLR of each model size used directly as a
predictor, and popDMS with a scale-matched LLR prior swept over prior strength and
regularization. Prior strength is expressed as a multiple of the **matched scale**
`s* = std(regular-popDMS coefficients) / std(LLR)`, so `1` means the prior's
coefficient spread equals popDMS's own.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

AGGREGATE_DIR = Path(os.environ.get("ESMDMS_AGGREGATE_DIR", "results/model_size_llr/aggregate"))
missing = [f for f in ("baselines.csv", "prior_sweeps.csv", "summary.csv")
           if not (AGGREGATE_DIR / f).is_file()]
if missing:
    raise RuntimeError(
        f"Missing {missing} under {AGGREGATE_DIR}. Run "
        "experiments/model_size_llr/submit_experiment.sh first, or set "
        "ESMDMS_AGGREGATE_DIR to an existing aggregate directory."
    )

baselines = pd.read_csv(AGGREGATE_DIR / "baselines.csv")
sweeps = pd.read_csv(AGGREGATE_DIR / "prior_sweeps.csv")
summary = pd.read_csv(AGGREGATE_DIR / "summary.csv")

PRIMARY_CUTOFF = 0
AUC = f"auc_stars_{PRIMARY_CUTOFF}"
DATASETS = sorted(baselines["dataset"].unique())
MODEL_SIZES = ["300M", "600M", "6B"]

# Ordinal model size -> single-hue blue ramp (light = small, dark = large).
SIZE_COLOR = {"300M": "#9ecae1", "600M": "#4292c6", "6B": "#08519c"}
# Colorblind-safe (Okabe-Ito) accents for the non-ESM baselines.
BASELINE_COLOR = {
    "Enrichment ratio": "#999999",
    "DMS functional score": "#000000",
    "Regular popDMS": "#E69F00",
}

def size_of(label):
    # "ESM-C 300M LLR" / "Raw ESM-C 300M LLR" -> "300M"
    for token in str(label).split():
        if token in SIZE_COLOR:
            return token
    return None

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
                     "figure.dpi": 110, "font.size": 10})
print(f"Loaded {len(DATASETS)} datasets from {AGGREGATE_DIR}")
summary

## Predictor comparison by dataset (ClinVar AUC)

One panel per dataset. Bars are the assay-oriented ClinVar AUC at review-star
cutoff 0. Color encodes ESM-C model size (blue ramp); **hatched** bars are the raw
LLR used directly, **solid** ESM-size bars are popDMS with that model's
scale-matched prior (best over the sweep). Grey/orange/black are the non-ESM
baselines. The dashed line is chance (0.5).

In [ ]:
best_prior = sweeps.groupby(["dataset", "prior"])[AUC].max()

def dataset_bars(dataset):
    rows, colors, hatches = [], [], []
    bl = baselines[baselines["dataset"] == dataset].set_index("method")[AUC]
    for method in ("Enrichment ratio", "DMS functional score", "Regular popDMS"):
        if method in bl.index and np.isfinite(bl[method]):
            rows.append((method, bl[method])); colors.append(BASELINE_COLOR[method]); hatches.append("")
    for size in MODEL_SIZES:
        raw = f"Raw ESM-C {size} LLR"
        if raw in bl.index and np.isfinite(bl[raw]):
            rows.append((f"Raw LLR {size}", bl[raw])); colors.append(SIZE_COLOR[size]); hatches.append("////")
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        if (dataset, prior) in best_prior.index:
            rows.append((f"Prior popDMS {size}", best_prior[(dataset, prior)]))
            colors.append(SIZE_COLOR[size]); hatches.append("")
    return rows, colors, hatches

ncol = 2
nrow = int(np.ceil(len(DATASETS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
for ax, dataset in zip(axes.flat, DATASETS):
    rows, colors, hatches = dataset_bars(dataset)
    labels = [r[0] for r in rows]; values = [r[1] for r in rows]
    x = np.arange(len(rows))
    bars = ax.bar(x, values, color=colors, width=0.72, edgecolor="white")
    for bar, h in zip(bars, hatches):
        if h: bar.set_hatch(h)
    ax.axhline(0.5, color="#C0392B", ls="--", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylim(0, 1); ax.set_ylabel("ClinVar AUC")
    ax.set_title(dataset, fontsize=10)
for ax in axes.flat[len(DATASETS):]:
    ax.set_visible(False)
size_handles = [Patch(facecolor=SIZE_COLOR[s], label=f"ESM-C {s}") for s in MODEL_SIZES]
type_handles = [Patch(facecolor="#cccccc", hatch="////", label="raw LLR"),
                Patch(facecolor="#cccccc", label="prior popDMS (best)")]
fig.legend(handles=size_handles + type_handles, loc="upper center",
           ncol=5, bbox_to_anchor=(0.5, 1.02), fontsize=9, frameon=False)
fig.tight_layout(rect=(0, 0, 1, 0.98))
plt.show()

## Correlation with the assay functional score (Spearman ρ)

Only datasets and methods with a finite functional score appear (BRCA2 has none).
Same color/hatch scheme as above.

In [ ]:
def spearman_rows(dataset):
    bl = baselines[baselines["dataset"] == dataset].set_index("method")["spearman_rho"]
    order, colors, hatches = [], [], []
    for method in ("Enrichment ratio", "Regular popDMS"):
        if method in bl.index and np.isfinite(bl[method]):
            order.append((method, bl[method])); colors.append(BASELINE_COLOR[method]); hatches.append("")
    for size in MODEL_SIZES:
        raw = f"Raw ESM-C {size} LLR"
        if raw in bl.index and np.isfinite(bl[raw]):
            order.append((f"Raw LLR {size}", bl[raw])); colors.append(SIZE_COLOR[size]); hatches.append("////")
    return order, colors, hatches

scored = [d for d in DATASETS
          if baselines[(baselines["dataset"] == d)]["spearman_rho"].notna().any()]
if not scored:
    print("No functional scores available in this result set.")
else:
    fig, axes = plt.subplots(1, len(scored), figsize=(3.6 * len(scored), 3.6), squeeze=False)
    for ax, dataset in zip(axes.flat, scored):
        rows, colors, hatches = spearman_rows(dataset)
        x = np.arange(len(rows))
        bars = ax.bar(x, [r[1] for r in rows], color=colors, edgecolor="white", width=0.7)
        for bar, h in zip(bars, hatches):
            if h: bar.set_hatch(h)
        ax.axhline(0, color="#666", lw=0.8)
        ax.set_xticks(x); ax.set_xticklabels([r[0] for r in rows], rotation=45, ha="right", fontsize=8)
        ax.set_ylabel("Spearman ρ vs functional score"); ax.set_title(dataset, fontsize=10)
    fig.tight_layout(); plt.show()

## Does the prior help, and at what strength?

For each dataset and model size, the best achievable AUC (over gamma) at each prior
strength, on a log2 axis of the **matched-scale multiple** (`1` = matched to the
popDMS coefficient spread). The grey dashed line is the `alpha = 0` no-prior
control; the square marker is the **unscaled raw-LLR** magnitude. A prior that helps
rises above its control line somewhere near `1`.

In [ ]:
scaled = sweeps[(sweeps["scale_multiple"] > 0) & (~sweeps["unscaled_raw_llr"])]
best_scaled = scaled.groupby(["dataset", "prior", "scale_multiple"])[AUC].max().reset_index()
controls = sweeps[sweeps["alpha"] == 0.0].groupby(["dataset", "prior"])[AUC].max()
unscaled = sweeps[sweeps["unscaled_raw_llr"]].groupby(["dataset", "prior"]).agg(
    scale_multiple=("scale_multiple", "first"), auc=(AUC, "max"))

ncol = 2
nrow = int(np.ceil(len(DATASETS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.4 * nrow), squeeze=False)
for ax, dataset in zip(axes.flat, DATASETS):
    for size in MODEL_SIZES:
        prior = f"ESM-C {size} LLR"
        line = best_scaled[(best_scaled["dataset"] == dataset) & (best_scaled["prior"] == prior)]
        if line.empty:
            continue
        line = line.sort_values("scale_multiple")
        ax.plot(line["scale_multiple"], line[AUC], marker="o", color=SIZE_COLOR[size],
                lw=2, label=f"ESM-C {size}")
        if (dataset, prior) in controls.index:
            ax.axhline(controls[(dataset, prior)], color=SIZE_COLOR[size], ls="--", lw=1, alpha=0.7)
        if (dataset, prior) in unscaled.index:
            u = unscaled.loc[(dataset, prior)]
            ax.scatter(u["scale_multiple"], u["auc"], color=SIZE_COLOR[size],
                       marker="s", s=55, zorder=5, edgecolor="white")
    ax.set_xscale("log", base=2)
    ax.axhline(0.5, color="#C0392B", ls=":", lw=1)
    ax.set_xlabel("prior strength (× matched scale s*)"); ax.set_ylabel("best ClinVar AUC")
    ax.set_title(dataset, fontsize=10); ax.legend(fontsize=8, title="dashed = no-prior control")
for ax in axes.flat[len(DATASETS):]:
    ax.set_visible(False)
fig.tight_layout(); plt.show()

## Regularization sensitivity (alpha × gamma surface)

The full surface behind the summary: best AUC as a function of gamma
(regularization / prior precision), one line per matched-scale multiple, per dataset
and model size.

In [ ]:
for dataset in DATASETS:
    priors = sorted(sweeps[sweeps["dataset"] == dataset]["prior"].unique())
    if not priors:
        continue
    fig, axes = plt.subplots(1, len(priors), figsize=(4.6 * len(priors), 3.6), squeeze=False)
    for ax, prior in zip(axes.flat, priors):
        frame = sweeps[(sweeps["dataset"] == dataset) & (sweeps["prior"] == prior)]
        surface = frame.pivot_table(index="gamma", columns="scale_multiple", values=AUC)
        cmap = plt.cm.viridis(np.linspace(0, 1, surface.shape[1]))
        for color, col in zip(cmap, surface.columns):
            ax.plot(surface.index, surface[col], color=color, lw=1.4,
                    label=f"{col:g}×" if col > 0 else "0")
        ax.set_xscale("log"); ax.axhline(0.5, color="#C0392B", ls=":", lw=1)
        ax.set_xlabel("gamma"); ax.set_ylabel("ClinVar AUC")
        ax.set_title(f"{dataset}\n{prior}", fontsize=9)
        ax.legend(fontsize=7, title="× s*", ncol=2)
    fig.tight_layout(); plt.show()

## Selected regularization and matched scale

For reproducibility: the popDMS elbow gamma chosen for regular popDMS, and the
matched scale `s*` with the coefficient/prior standard deviations that define it,
per dataset and model size.

In [ ]:
elbow = (baselines[baselines["method"] == "Regular popDMS"]
         .set_index("dataset")["gamma"].rename("elbow_gamma"))
scale_tbl = (sweeps.groupby(["dataset", "prior"])
             .agg(matched_scale=("matched_scale", "first"),
                  sigma_coeff=("sigma_coeff", "first"),
                  sigma_prior=("sigma_prior", "first"))
             .reset_index())
scale_tbl["model_size"] = scale_tbl["prior"].map(size_of)
scale_tbl = scale_tbl.merge(elbow, left_on="dataset", right_index=True, how="left")
scale_tbl[["dataset", "model_size", "elbow_gamma", "matched_scale",
           "sigma_coeff", "sigma_prior"]].sort_values(["dataset", "model_size"])